# ch01 时序数据与特征工程（练习版）

> 补全所有 `____`，让每个代码块末尾的 `assert` 全绿。
> 数据：`data/load_curve.csv`（2 台区 × 8760 h）+ `data/load_curve_messy.csv`（336 行脏快照）。
>
> **每题都跟了 `# 提示：`**，照着提示能独立做完。
> 如果 assert 的数字对不上，**先回头看上一步的输出**，别急着改 assert —— 那些数字都是实跑出来的。

## 任务清单

| 任务 | 主题 | 挖空 | 对应讲解小节 |
|---|---|---|---|
| 1 | 规则时间轴（去重 → 排序 → 对齐） | 2 | §1.1 |
| 2 | 混合时间戳 + 脏数值列 | 2 | §1.1.1 |
| 3 | 时间特征 + 节假日标记 | 2 | §1.2 |
| 4 | sin/cos 周期编码 | 2 | §1.2.1 |
| 5 | 组内 / 全局 `shift` | 2 | §1.3.1 |
| 6 | 清洗与 `rolling` 的 NaN 传播 | 2 | §1.3.2 |
| 7 | 监督矩阵组装 | 2 | §1.4 |
| 8 | 三种泄漏实测 | 4 | §1.4.1 |
| 9 | 三层周期分解 | 2 | §1.5 |
| 10 | 加法 vs 对数 | 3 | §1.5.1 |
| 11 | 残差 ACF 诊断 | 3 | §1.5.2 |
| 12 | 多序列对齐 + 温度 V 形 | 3 | §1.6 |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

DATA = Path("data")          # notebook 的 cwd = coding/05_timeseries
CSV = DATA / "load_curve.csv"
MESSY_CSV = DATA / "load_curve_messy.csv"

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

raw = pd.read_csv(CSV, encoding="utf-8-sig", parse_dates=["时间戳"])
print("原始行数", len(raw), "| 列", list(raw.columns))
print("台区", sorted(raw["台区编号"].unique()))
print("时间跨度", raw["时间戳"].min(), "~", raw["时间戳"].max())
print("原始时间戳单调递增？", raw["时间戳"].is_monotonic_increasing)

In [ ]:
# 规则化的第一步：去重 + 按时间排序。
# **顺序不能反** —— 先用 asfreq 再排序，规则时间轴会被重复时刻搞乱（见 1.1.1）
df = raw.drop_duplicates().sort_values("时间戳").reset_index(drop=True)
print("去重排序后", len(df), "行 | 单调递增", df["时间戳"].is_monotonic_increasing)
print("负荷值缺失", int(df["负荷值"].isna().sum()),
      "| 温度缺失", int(df["温度"].isna().sum()))
print(df.head(3))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# macOS 上 matplotlib 默认字体不含中文，不设置的话图上全是方块
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib", matplotlib.__version__, "| 中文字体 PingFang SC")

In [ ]:
# 脚手架：下面的常量与工具函数两版都有，你只填 @@todo 块里的内容
HOLIDAY_RANGES = [
    ("2025-01-01", "2025-01-01"),   # 元旦
    ("2025-01-28", "2025-02-04"),   # 春节
    ("2025-04-04", "2025-04-06"),   # 清明
    ("2025-05-01", "2025-05-05"),   # 劳动节
    ("2025-05-31", "2025-06-02"),   # 端午
    ("2025-10-01", "2025-10-08"),   # 中秋 + 国庆
]
HOLIDAYS = set()
for _lo, _hi in HOLIDAY_RANGES:
    HOLIDAYS |= set(pd.date_range(_lo, _hi, freq="D").strftime("%Y-%m-%d"))


def clean_series(s):
    """按**小时分组**的中位数做 3 倍阈值判异常，再线性插值。

    为什么不能直接用「全局中位数 × 3」：工业台区是「夜间低 + 日间高」的双峰
    分布，全局阈值会把整段日间峰误判成异常（实测 B 台区会误杀 382 个点）。
    """
    med = s.groupby(s.index.hour).transform("median")
    return s.where(s.between(0, 3 * med)).interpolate()


def cyc(values, period):
    """把「周期整数」编码成 (sin, cos) 两列，原理见 1.2.1。"""
    ang = 2 * np.pi * np.asarray(values, dtype=float) / period
    return np.sin(ang), np.cos(ang)


def acf(s, lag):
    """自相关（转发 pandas 内置实现，便于阅读）。"""
    return float(s.autocorr(lag))

## 任务 1：把 CSV 变成规则时间轴（§1.1）

In [ ]:
a_frame = df[df["台区编号"] == "STATION_A_01"].set_index("时间戳").sort_index()
b_frame = df[df["台区编号"] == "STATION_B_02"].set_index("时间戳").sort_index()

s_a = a_frame["负荷值"].asfreq("h")

s_b = b_frame["负荷值"].asfreq("h")

print("A 台区 %d 个整点 | 缺失 %d | 完整率 %.6f"
      % (len(s_a), int(s_a.isna().sum()), 1 - s_a.isna().mean()))
print("B 台区 %d 个整点 | 缺失 %d | 完整率 %.6f"
      % (len(s_b), int(s_b.isna().sum()), 1 - s_b.isna().mean()))

# ---- 验收 ----
assert len(s_a) == 8760 and len(s_b) == 8760, "两个台区都应是 8760 个整点"
assert int(s_a.isna().sum()) == 70, f"A 台区应缺 70 个，实际 {int(s_a.isna().sum())}"
assert int(s_b.isna().sum()) == 79, f"B 台区应缺 79 个，实际 {int(s_b.isna().sum())}"
assert isinstance(s_a.index, pd.DatetimeIndex) and s_a.index.freq is not None

## 任务 2：解析混合格式时间戳 + 脏数值列（§1.1.1）

In [ ]:
messy = pd.read_csv(MESSY_CSV, encoding="utf-8-sig")
print("列名（注意两侧带空格）:", list(messy.columns))
print("时间戳前 4 行:", messy[" 时间戳 "].head(4).tolist())

ts_text = (messy[" 时间戳 "]
           .str.replace("年", "-", regex=False).str.replace("月", "-", regex=False)
           .str.replace("日", " ", regex=False).str.replace("时", ":00", regex=False))
ts_dt = pd.to_datetime(ts_text, format="mixed")

load_num = pd.to_numeric(messy["负荷值(kW)"].str.replace(",", "", regex=False),
                         errors="coerce")

print("解析后 NaT %d | 负荷列 NaN %d | 哨兵 9999 %d"
      % (int(ts_dt.isna().sum()), int(load_num.isna().sum()), int((load_num == 9999).sum())))

# ---- 验收 ----
assert len(messy) == 336
assert int(ts_dt.isna().sum()) == 0, "336 行时间戳应全部解析成功"
assert int(load_num.isna().sum()) == 8, "去掉千分位后仍有 8 个脏文本（--/N/A/空串）"
assert int((load_num == 9999).sum()) == 3

## 任务 3：派生时间特征与节假日标记（§1.2）

In [ ]:
feat = pd.DataFrame(index=s_a.index)

feat["hour"] = s_a.index.hour
feat["dow"] = s_a.index.dayofweek
feat["month"] = s_a.index.month
feat["doy"] = s_a.index.dayofyear

feat["is_holiday"] = s_a.index.strftime("%Y-%m-%d").isin(HOLIDAYS).astype(int)

feat["is_weekend"] = (s_a.index.dayofweek >= 5).astype(int)

print(feat.head(3))
print("节假日小时 %d | 周末小时 %d"
      % (int(feat["is_holiday"].sum()), int(feat["is_weekend"].sum())))

# ---- 验收 ----
assert int(feat["is_holiday"].sum()) == 672, "2025 年法定节假日 28 天 × 24 = 672 小时"
assert int(feat["is_weekend"].sum()) == 2496
assert int(feat["doy"].max()) == 365 and int(feat["hour"].max()) == 23

## 任务 4：周期整数的 sin/cos 编码（§1.2.1）

In [ ]:
feat["hour_sin"], feat["hour_cos"] = cyc(s_a.index.hour, 24)

feat["dow_sin"], feat["dow_cos"] = cyc(s_a.index.dayofweek, 7)

d_raw = abs(23 - 0)
d_cyc = float(np.hypot(feat["hour_sin"].iloc[23] - feat["hour_sin"].iloc[0],
                       feat["hour_cos"].iloc[23] - feat["hour_cos"].iloc[0]))
d_12 = float(np.hypot(feat["hour_sin"].iloc[12] - feat["hour_sin"].iloc[0],
                      feat["hour_cos"].iloc[12] - feat["hour_cos"].iloc[0]))
print("23 点 vs 0 点：原始距离 %d → 编码后弦长 %.6f" % (d_raw, d_cyc))
print("对照 0 点 vs 12 点：原始距离 12 → 编码后弦长 %.6f" % d_12)
print("理论值 2*sin(pi/24) = %.6f | 2*sin(pi/2) = %.6f"
      % (2 * np.sin(np.pi / 24), 2 * np.sin(np.pi / 2)))

# ---- 验收 ----
assert d_raw == 23
assert abs(d_cyc - 2 * np.sin(np.pi / 24)) < 1e-12, "23h 与 0h 在编码空间只差 0.261"
assert abs(d_12 - 2.0) < 1e-12, "0h 与 12h 恰好相差一个直径"

## 任务 5：组内 shift 与全局 shift 的差别（§1.3.1）

In [ ]:
d2 = df.sort_values(["台区编号", "时间戳"]).reset_index(drop=True)

d2["lag_bad"] = d2["负荷值"].shift(1)

d2["lag_ok"] = d2.groupby("台区编号")["负荷值"].shift(1)

a_last = float(d2[d2["台区编号"] == "STATION_A_01"]["负荷值"].iloc[-1])
print("A 台区最后一行负荷            : %.2f" % a_last)
print("B 台区首行 全局 shift 取到     : %.2f" % float(d2["lag_bad"].iloc[8760]))
print("B 台区首行 组内 shift 取到     : %s" % d2["lag_ok"].iloc[8760])
print("组内 shift 的 NaN 总数         : %d" % int(d2["lag_ok"].isna().sum()))

# ---- 验收 ----
assert abs(float(d2["lag_bad"].iloc[8760]) - a_last) < 1e-9, "全局 shift 把 A 的末行喂给了 B"
assert pd.isna(d2["lag_ok"].iloc[8760]), "组内 shift 的首行必须是 NaN"
assert int(d2["lag_ok"].isna().sum()) == 151, "149 个原始缺失移位 + 2 个组首行"

## 任务 6：清洗，以及 rolling 的 NaN 传播（§1.3.2）

In [ ]:
s_a_clean = clean_series(s_a)

print("1 步自相关：未清洗 %.6f → 清洗后 %.6f" % (acf(s_a, 1), acf(s_a_clean, 1)))

nan_raw = int(s_a.shift(1).rolling(24).mean().isna().sum())
nan_clean = int(s_a_clean.shift(1).rolling(24).mean().isna().sum())

print("rolling(24).mean() 的 NaN：未清洗 %d | 清洗后 %d" % (nan_raw, nan_clean))

# ---- 验收 ----
assert nan_raw == 1009, "只有 70 个缺失，却污染了 1009 行——NaN 沿窗口向后传播"
assert nan_clean == 24, "清洗后只剩开头的窗口不完整导致的 24 行"

## 任务 7：组装监督学习矩阵（§1.4）

In [ ]:
w = pd.DataFrame(index=s_a_clean.index)
w["y"] = s_a_clean.shift(-1)         # 目标：下一小时的负荷
prev = s_a_clean.shift(1)            # 上一时刻（用来构造滑窗，天然不含未来）

w["lag1"] = s_a_clean.shift(1)
w["lag2"] = s_a_clean.shift(2)
w["lag24"] = s_a_clean.shift(24)

w["ma24"] = prev.rolling(24).mean()
w["std24"] = prev.rolling(24).std()

w["hour_sin"], w["hour_cos"] = cyc(w.index.hour, 24)
w["dow_sin"], w["dow_cos"] = cyc(w.index.dayofweek, 7)

FEAT = ["lag1", "lag2", "lag24", "ma24", "std24",
        "hour_sin", "hour_cos", "dow_sin", "dow_cos"]
model = w[FEAT + ["y"]].dropna()
print("特征数 %d | dropna 前 %d 行 → 后 %d 行（损失 %d）"
      % (len(FEAT), len(w), len(model), len(w) - len(model)))
print("lag1  与 y 相关 %.6f" % float(model["lag1"].corr(model["y"])))
print("lag24 与 y 相关 %.6f  ← 比 lag1 还高" % float(model["lag24"].corr(model["y"])))
print("ma24  与 y 相关 %.6f" % float(model["ma24"].corr(model["y"])))

# ---- 验收 ----
assert len(w) - len(model) == 25, "lag24 吃掉 24 行 + y 的尾部 1 行"
assert abs(float(model["lag1"].corr(model["y"])) - 0.653541) < 1e-6
assert abs(float(model["lag24"].corr(model["y"])) - 0.826982) < 1e-6

## 任务 8：三种泄漏实测（§1.4.1）

In [ ]:
cut = int(len(w) * 0.8)
tr = w.iloc[:cut][FEAT + ["y"]].dropna()
te = w.iloc[cut:][FEAT + ["y"]].dropna()
print("切分点 %d | 时间 %s | 训练 %d / 测试 %d"
      % (cut, w.index[cut], len(tr), len(te)))
print("训练段 y 均值 %.4f | 测试段 y 均值 %.4f" % (tr["y"].mean(), te["y"].mean()))

scaler = StandardScaler().fit(tr[FEAT])
base = Ridge(alpha=1.0).fit(scaler.transform(tr[FEAT]), tr["y"])
mae_base = mean_absolute_error(te["y"], base.predict(scaler.transform(te[FEAT])))
print("① 干净基线 测试 MAE %.6f" % mae_base)

w["fut_ma24"] = s_a_clean.shift(-1).rolling(24).mean()

FEAT_LEAK = FEAT + ["fut_ma24"]
tr2 = w.iloc[:cut][FEAT_LEAK + ["y"]].dropna()
te2 = w.iloc[cut:][FEAT_LEAK + ["y"]].dropna()
sc2 = StandardScaler().fit(tr2[FEAT_LEAK])
leaked = Ridge(alpha=1.0).fit(sc2.transform(tr2[FEAT_LEAK]), tr2["y"])
mae_leak = mean_absolute_error(te2["y"], leaked.predict(sc2.transform(te2[FEAT_LEAK])))
print("② 加 fut_ma24 测试 MAE %.6f（%+.6f）← 指标变好，是假的"
      % (mae_leak, mae_leak - mae_base))

rand_tr = model.sample(frac=0.8, random_state=42)
rand_te = model.drop(rand_tr.index)

sc_r = StandardScaler().fit(rand_tr[FEAT])
m_r = Ridge(alpha=1.0).fit(sc_r.transform(rand_tr[FEAT]), rand_tr["y"])
mae_rand = mean_absolute_error(rand_te["y"], m_r.predict(sc_r.transform(rand_te[FEAT])))

mdl_x = model.iloc[: int(len(model) * 0.8)][FEAT]
sc_t = StandardScaler().fit(mdl_x)
m_t = Ridge(alpha=1.0).fit(sc_t.transform(mdl_x), model.iloc[: int(len(model) * 0.8)]["y"])
mae_time = mean_absolute_error(
    model.iloc[int(len(model) * 0.8):]["y"],
    m_t.predict(sc_t.transform(model.iloc[int(len(model) * 0.8):][FEAT])))
print("③ 随机切分 MAE %.6f vs 时间切分 MAE %.6f（差 %+.6f）"
      % (mae_rand, mae_time, mae_rand - mae_time))

sc_all = StandardScaler().fit(w[FEAT].dropna())
m_all = Ridge(alpha=1.0).fit(sc_all.transform(tr[FEAT]), tr["y"])
mae_scaler = mean_absolute_error(te["y"], m_all.predict(sc_all.transform(te[FEAT])))

print("④ 全量标准化 MAE %.6f（%+.6f）← 几乎没动"
      % (mae_scaler, mae_scaler - mae_base))
print("   lag1 全量均值 %.4f vs 训练段均值 %.4f（差 %.4f）"
      % (w["lag1"].mean(), tr["lag1"].mean(), abs(w["lag1"].mean() - tr["lag1"].mean())))

# ---- 验收 ----
assert abs(mae_base - 26.797903) < 1e-5
assert abs(mae_leak - 19.958735) < 1e-5 and mae_leak < mae_base
assert abs(mae_rand - 25.643746) < 1e-5 and mae_rand < mae_time
assert abs(mae_scaler - 26.797910) < 1e-5, "标准化泄漏在本数据上几乎无影响——这是负结果"

## 任务 9：手写三层周期分解（§1.5）

In [ ]:
s = s_a_clean

trend = s.rolling(24, center=True, min_periods=24).mean().bfill().ffill()

detr = s - trend

season = detr.groupby(detr.index.hour).transform("mean")

resid = detr - season
recon_err = float((trend + season + resid - s).abs().max())
tr_by_m = trend.groupby(trend.index.month).mean()
print("重建最大绝对误差 %.3e（三层相加必然还原原序列）" % recon_err)
print("低频项 月均 peak/valley %.4f / %.4f（月 %d / %d）"
      % (tr_by_m.max(), tr_by_m.min(), int(tr_by_m.idxmax()), int(tr_by_m.idxmin())))
print("日周期项 peak/valley %.4f / %.4f（hour %d / %d）"
      % (season.max(), season.min(),
         int(season.groupby(season.index.hour).mean().idxmax()),
         int(season.groupby(season.index.hour).mean().idxmin())))
print("残差 std %.6f | 残差 std / 序列均值 %.6f" % (resid.std(), resid.std() / s.mean()))

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
n_show = 24 * 21
axes[0].plot(s.index[:n_show], s.iloc[:n_show], lw=0.8, label="原序列")
axes[0].plot(trend.index[:n_show], trend.iloc[:n_show], lw=1.6, label="低频项（24h 居中均值）")
axes[0].set_ylabel("负荷 kW")
axes[0].legend(loc="upper right")
axes[1].plot(detr.index[:n_show], detr.iloc[:n_show], lw=0.6, color="gray", label="去趋势")
axes[1].plot(season.index[:n_show], season.iloc[:n_show], lw=1.6, color="tab:orange",
             label="日周期项")
axes[1].set_ylabel("去趋势后")
axes[1].legend(loc="upper right")
axes[2].plot(resid.index[:n_show], resid.iloc[:n_show], lw=0.6, color="crimson")
axes[2].set_ylabel("残差")
axes[2].set_xlabel("时间")
plt.tight_layout()

# ---- 验收 ----
assert recon_err < 1e-9
assert abs(float(season.max()) - 121.8327) < 1e-3
assert abs(float(season.min()) - (-60.1567)) < 1e-3
assert abs(float(resid.std()) - 20.673831) < 1e-5

## 任务 10：加法分解还是对数分解（§1.5.1）

In [ ]:
ls = np.log(s)
print("取对数前 min/max %.2f / %.2f（必须全 > 0 才能取对数）" % (s.min(), s.max()))

l_trend = ls.rolling(24, center=True, min_periods=24).mean().bfill().ffill()

l_detr = ls - l_trend
l_resid = l_detr - l_detr.groupby(l_detr.index.hour).transform("mean")

add_ratio = float(resid[s >= s.quantile(0.5)].std() / resid[s < s.quantile(0.5)].std())
log_ratio = float(l_resid[ls >= ls.quantile(0.5)].std() / l_resid[ls < ls.quantile(0.5)].std())

print("加法：残差 std %.6f | 高/低段比 %.6f  ← 高负荷段波动明显更大" % (resid.std(), add_ratio))
print("对数：残差 std %.6f | 高/低段比 %.6f  ← 两段几乎相等" % (l_resid.std(), log_ratio))
print("换算成相对残差 expm1(%.6f) = %.6f" % (l_resid.std(), float(np.expm1(l_resid.std()))))

# ---- 验收 ----
assert abs(add_ratio - 1.171187) < 1e-5, "加法分解的残差方差随负荷水平增长 17%"
assert abs(log_ratio - 1.011203) < 1e-5, "对数分解后残差方差不再随水平变化"

## 任务 11：用残差 ACF 诊断分解是否到位（§1.5.2）

In [ ]:
LAGS = (1, 24, 168)

acf_add = [round(acf(resid, k), 4) for k in LAGS]
acf_log = [round(acf(l_resid, k), 4) for k in LAGS]

print("lag            ", "  ".join("lag%-6d" % k for k in LAGS))
print("加法残差 ACF   ", "  ".join("%+.4f " % v for v in acf_add))
print("对数残差 ACF   ", "  ".join("%+.4f " % v for v in acf_log))

amp = (l_detr.groupby([l_detr.index.month, l_detr.index.hour]).mean()
       .groupby(level=0).apply(lambda x: x.max() - x.min()))

print("日周期幅度按月（对数域）:", {int(k): round(float(v), 4) for k, v in amp.items()})

season2 = l_detr.groupby([l_detr.index.month, l_detr.index.hour]).transform("mean")
resid2 = l_detr - season2

print("季节项改 (月,小时) 后：残差 std %.6f → %.6f | ACF(1) %+.4f → %+.4f"
      % (l_resid.std(), resid2.std(), acf(l_resid, 1), acf(resid2, 1)))

# ---- 验收 ----
assert abs(float(amp.loc[7]) - 0.4567) < 1e-3, "7 月的日周期幅度最大"
assert abs(float(amp.loc[1]) - 0.3496) < 1e-3, "1 月的日周期幅度最小"
assert abs(resid2.std() - 0.034551) < 1e-6
assert acf(resid2, 24) < 0.05, "换成 (月,小时) 后 lag24 的自相关基本消失"

## 任务 12：多序列对齐 + 温度的非线性（§1.6）

In [ ]:
wide = df.pivot(index="时间戳", columns="台区编号", values="负荷值")

s_b_clean = clean_series(s_b)
pair = pd.DataFrame({"A": s_a_clean, "B": s_b_clean})
print("宽表 shape %s | 两列全 NaN 的行 %d" % (wide.shape, int(wide.isna().all(axis=1).sum())))

pair_h = pair.groupby(pair.index.hour).mean()
shape_corr = float(pair_h.corr().iloc[0, 1])

print("A-B 同时刻相关 %.6f（几乎不相关）" % float(s_a_clean.corr(s_b_clean)))
print("A-B 日周期形状相关 %.6f（负相关：晚高峰 vs 日间平台）" % shape_corr)

temp = (df[df["台区编号"] == "STATION_A_01"]
        .set_index("时间戳").sort_index()["温度"].asfreq("h").interpolate())

lin = float(s_a_clean.corr(temp))
vshape = float((temp - 22).abs().corr(s_a_clean))

bins = pd.cut(temp, [-10, 0, 10, 20, 26, 32, 40])
print("线性相关 %.6f | |T-22| 相关 %.6f" % (lin, vshape))
print("温度分箱后的负荷均值:",
      {str(k): round(float(v), 1) for k, v in s_a_clean.groupby(bins).mean().items()})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(pair_h.index, pair_h["A"], marker="o", label="A 居民型")
axes[0].plot(pair_h.index, pair_h["B"], marker="s", label="B 工业型")
axes[0].set_xlabel("小时")
axes[0].set_ylabel("平均负荷 kW")
axes[0].legend()
axes[1].plot(temp, s_a_clean, ".", ms=1, alpha=0.2)
axes[1].set_xlabel("温度 ℃")
axes[1].set_ylabel("负荷 kW")
axes[1].set_title("温度—负荷：V 形")
plt.tight_layout()

# ---- 验收 ----
assert wide.shape == (8760, 2)
assert abs(shape_corr - (-0.040653)) < 1e-6
assert abs(lin - (-0.188929)) < 1e-6
assert abs(vshape - 0.406437) < 1e-6, "换成 V 形度量后相关度从 0.19 涨到 0.41"

## 自查清单（能不看笔记答出来才算过）

- [ ] 12 个代码块的 assert 全部通过
- [ ] `drop_duplicates` 必须在哪一步之前？不这样做会怎样？
- [ ] `asfreq` 与 `resample` 的区别是什么？想"数出缺了几小时"该用哪个？
- [ ] 为什么 `hour` 要做 sin/cos 编码？23 点与 0 点编码后的距离是多少？
- [ ] B 台区首行用全局 `shift` 会取到哪个值？它为什么"看起来很合理"？
- [ ] 70 个缺失为什么会让 **1009** 行 `rolling(24)` 结果变成 NaN？
- [ ] 清洗异常值为什么不能直接用全局中位数？B 台区会被误杀多少个点？
- [ ] 三种泄漏分别是什么？哪一种在本数据上**测不出效果**，为什么？
- [ ] 怎么判断该用加法还是对数分解？给出那条可背的判据。
- [ ] 残差 ACF 平坦意味着什么？怎么修？
- [ ] 为什么温度要用 `|T − 22|` 而不是 `T`？
- [ ] `lag24` 为什么比 `lag1` 与 `y` 更相关？

## 关键真值对照（做完再对）

| 量 | 值 |
|---|---|
| 原始行数 / 去重后行数 | 17526 / 17520 |
| 完全重复行 | 6 |
| 原始时间戳单调递增？ | False → 排序后 True |
| A / B 台区缺失数 | 70 / 79 |
| 节假日小时 / 周末小时 | 672 / 2496 |
| 23↔0 编码弦长 | 0.261052 |
| 组内 shift 的 NaN 数 | 151 |
| B 台区首行被污染的值 | 497.96 |
| 1 步自相关：未清洗 / 清洗后 | 0.277454 / 0.868876 |
| `rolling(24)` NaN：未清洗 / 清洗后 | 1009 / 24 |
| 监督矩阵 dropna 损失 | 25 |
| `lag1` / `lag24` 与 `y` 相关 | 0.653541 / 0.826982 |
| 基线测试 MAE | 26.797903 |
| 加 `fut_ma24` 后 MAE | 19.958735 |
| 随机切分 / 时间切分 MAE | 25.643746 / 26.762002 |
| 全量标准化 MAE | 26.797910 |
| 日周期项 peak / valley | 121.8327 / −60.1567 |
| 加法高/低段残差 std 比 | 1.171187 |
| 对数高/低段残差 std 比 | 1.011203 |
| 7 月 / 1 月 日周期幅度 | 0.4567 / 0.3496 |
| `(月, 小时)` 分组后残差 std | 0.034551 |
| A-B 日周期形状相关 | −0.040653 |
| 温度线性 / V 形相关 | −0.188929 / 0.406437 |